# 15 · MLOps basics: track experiments with MLflow
After a dozen experiments nobody remembers which settings gave which score. **MLflow** records parameters, metrics and the model for every run so you can compare and reproduce. It is a real tool used in industry.

Needs: `pip install mlflow` (a large package). Runs are stored locally in `python_practice/_out/mlflow.db`; view them with `mlflow ui --backend-store-uri sqlite:///_out/mlflow.db` run from `python_practice/`.

In [ ]:
import sys
sys.path.insert(0, "..")          # lets the notebook import common.py from python_practice/
from common import *              # load_operations, check, OUT, ...
import numpy as np
import pandas as pd

df = load_operations()            # reads the dashboard API if it is running, else data_samples/operations_clean.csv
df["not_completed"] = (df["status"] != "Completed").astype(int)
import os, joblib, mlflow
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

mlflow.set_tracking_uri(f"sqlite:///{(OUT / 'mlflow.db').as_posix()}")
if mlflow.get_experiment_by_name("bi-revenue-regression") is None:   # keep artifacts under _out/, not next to the notebook
    mlflow.create_experiment("bi-revenue-regression", artifact_location=(OUT / "mlartifacts").as_posix())
mlflow.set_experiment("bi-revenue-regression")

In [ ]:
df["weekday"] = df["record_date"].dt.day_name()
NUM, CAT = ["units_processed", "duration_minutes"], ["category", "weekday"]
days = np.sort(df["record_date"].unique()); cutoff = days[int(len(days) * 0.8)]
train, test = df[df["record_date"] < cutoff], df[df["record_date"] >= cutoff]

## Log three runs
Each `with mlflow.start_run()` block is one experiment run. `log_param` records what you chose, `log_metric` what you got, `log_artifact` the fitted pipeline (saved with joblib).

In [ ]:
candidates = {"ridge": Ridge(), "random_forest": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=0),
              "gradient_boosting": GradientBoostingRegressor(random_state=0)}
scores = {}
for name, model in candidates.items():
    pre = ColumnTransformer([("num", StandardScaler(), NUM), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    pipe = Pipeline([("pre", pre), ("model", model)]).fit(train[NUM + CAT], train["revenue"])
    pred = pipe.predict(test[NUM + CAT])
    with mlflow.start_run(run_name=name):
        mlflow.log_param("model", name); mlflow.log_param("features", ",".join(NUM + CAT)); mlflow.log_param("split", "time, last 20% of days")
        mae, r2 = mean_absolute_error(test["revenue"], pred), r2_score(test["revenue"], pred)
        mlflow.log_metric("mae", mae); mlflow.log_metric("r2", r2)
        joblib.dump(pipe, f"{name}.joblib"); mlflow.log_artifact(f"{name}.joblib"); os.remove(f"{name}.joblib")
    scores[name] = mae
pd.Series(scores, name="MAE").round(3)

## Compare runs programmatically
`mlflow.search_runs` returns a DataFrame of every run in the experiment.

In [ ]:
runs = mlflow.search_runs(order_by=["metrics.mae ASC"])
runs[["tags.mlflow.runName", "params.model", "metrics.mae", "metrics.r2"]].head(10)

## Your turn
Find the **name** of the run with the lowest MAE using `search_runs` (the run name is in the `tags.mlflow.runName` column). Store it in `best_run_name`.

In [ ]:
best_run_name = None

In [ ]:
check("best_run_name", best_run_name == min(scores, key=scores.get), "mlflow.search_runs(order_by=['metrics.mae ASC']).iloc[0]['tags.mlflow.runName']")

**Next steps in a real project:** register the best model in the Model Registry, add a tag like `stage=candidate`, and serve it (`mlflow models serve`) behind an API. Always log the data version and code version too.